In [29]:
import gymnasium as gym  
import numpy as np
import pandas as pd
import torch
import torch.nn as nn 
from torch.distributions.normal import Normal
import random 
from tqdm import tqdm 
import torch.nn.functional as F


In [34]:
class Reinforce_agent:
    def __init__(self):
        self.env = gym.make("InvertedPendulum-v4")
        self.n_states = self.env.observation_space.shape[0]
        self.n_actions = self.env.action_space.shape[0]
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.gamma = 0.99
        self.lr = 0.0001
        self.eps = 1e-6

        self.probs = []
        self.rewards = []

    def create_policy_net(self):
        class Policy_Network(nn.Module):
            def __init__(self, n_actions, n_states):
                super(Policy_Network,self).__init__() 

                hidden_layer_1 = 16
                hidden_layer_2 = 32
                hidden_layer_3 = 16

                self.base_net= nn.Sequential(
                    nn.Linear(n_states,hidden_layer_1),
                    nn.Tanh(),
                    nn.Linear(hidden_layer_1,hidden_layer_2),
                    nn.Tanh(),
                    nn.Linear(hidden_layer_2,hidden_layer_3),
                    nn.Tanh(),
                )

                self.policy_mean_net = nn.Sequential(
                    nn.Linear(hidden_layer_3,n_actions)
                )
                self.policy_std_net = nn.Sequential(
                    nn.Linear(hidden_layer_3, n_actions)
                )
            def forward(self, states):
                base_nn_output = self.base_net(states.float())
                action_means = self.policy_mean_net(base_nn_output)
                # action_stds = torch.log(1 + torch.exp(self.policy_std_net(base_nn_output)))
                action_stds = F.softplus(self.policy_std_net(base_nn_output)) + 1e-3
                return action_means,action_stds
            
        self.policy_net = Policy_Network(self.n_actions,self.n_states).to(self.device)
        self.optimizer = torch.optim.Adam(self.policy_net.parameters(),lr=self.lr)

    def get_action(self, state):
        state = torch.from_numpy(state).float().unsqueeze(0).to(self.device)

        action_means, action_stds = self.policy_net(state)

        dist = Normal(action_means[0], action_stds[0])

        action = dist.sample()

        log_prob = dist.log_prob(action).sum()
        self.probs.append(log_prob)

        return action.detach().cpu().numpy()
    
    def update(self):
        running_g = 0   
        gs = []

        for R in self.rewards[::-1]:
            running_g = R+self.gamma *running_g
            gs.insert(0,running_g)
        deltas = torch.tensor(gs).to(self.device)
        deltas = (deltas - deltas.mean()) / (deltas.std() + self.eps)

        log_probs = torch.stack(self.probs).squeeze()
        loss = -torch.sum(log_probs*deltas).to(self.device)

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

        self.probs = []
        self.rewards = [] 

    def train(self,n_episodes = 5000, ):
        for seed in [1,2,3,4,5]:
            torch.manual_seed(seed)
            random.seed(seed)
            np.random.seed(seed)

        for i in tqdm(range(n_episodes)):

            state,info = self.env.reset(seed = seed)

            done = False
            while not done:
                action = self.get_action(state)
                state,reward,terminated, truncated,info = self.env.step(action)
                done = terminated or truncated
                self.rewards.append(reward)
            self.update()


    def save_model(self):
        torch.save(self.policy_net.state_dict(),"policy_agent")
    
    def load_model(self):
        self.policy_net.load_state_dict(torch.load("policy_agent"))
        self.policy_net.eval()


        


In [35]:
agent = Reinforce_agent()

In [36]:
agent.create_policy_net()

In [37]:
agent.train()

 70%|██████▉   | 3496/5000 [22:10<09:32,  2.63it/s]  


KeyboardInterrupt: 

In [9]:
agent.save_model()

In [15]:
agent.load_model()

C:\Users\User\AppData\Local\Temp\ipykernel_3136\2559905620.py:101: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.policy_net.load_state_dict(torch.load("policy_agent"))


In [16]:
env = gym.make("InvertedPendulum-v4",render_mode='human')
for i in range(10):
    done = False
    state , info = env.reset()

    while not done:
        action = agent.get_action(state)
        state, reward, terminated , truncated , info = env.step(action)
        done = terminated or truncated
env.close()

    

